# Paperless-ngx ML Data Platform — Chameleon Setup

Run this notebook in the **Chameleon Jupyter environment** to:
1. Provision a VM on KVM@TACC and bring up the data platform
2. Create a **persistent** object storage bucket on CHI@TACC
3. Generate S3 credentials for the bucket
4. Run the ingestion pipeline to populate the persistent bucket

# Part 1: VM Setup

## Step 1 — Select project and site

In [ ]:
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
print(f"Username: {username}")

## Step 2 — Reserve VM (8 hours)

In [ ]:
l = lease.Lease(
    f"lease-paperless-{username}",
    duration=datetime.timedelta(hours=8)
)
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.xlarge"), amount=1)
l.submit(idempotent=True)
l.show()

## Step 3 — Launch VM instance

In [ ]:
s = server.Server(
    f"node-paperless-{username}",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

## Step 4 — Assign floating IP

In [ ]:
s.associate_floating_ip()
s.refresh()
s.show(type="widget")

## Step 5 — Open security groups

In [ ]:
security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-5050", "port": 5050, "description": "Adminer UI"},
    {"name": "allow-9001", "port": 9001, "description": "MinIO Console"},
    {"name": "allow-8090", "port": 8090, "description": "Redpanda Console"},
    {"name": "allow-6333", "port": 6333, "description": "Qdrant"},
    {"name": "allow-8080", "port": 8080, "description": "Airflow UI"},
    {"name": "allow-8000", "port": 8000, "description": "Stub API"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Security groups applied: {[sg['name'] for sg in security_groups]}")

## Step 6 — Install Docker

In [ ]:
s.refresh()
s.check_connectivity()
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
print("Docker installed.")

## Step 7 — Clone repo and bring up the platform

In [ ]:
REPO_URL = "https://github.com/REDES01/paperless_data.git"

s.execute(f"git clone {REPO_URL} ~/paperless_data")
s.execute("cd ~/paperless_data && sg docker -c 'docker compose -f docker/docker-compose.yaml up -d'")
print("Platform started.")

## Step 8 — Print access URLs

In [ ]:
s.refresh()
addresses = s.addresses
floating_ip = None
for net, addrs in addresses.items():
    for addr in addrs:
        if addr.get("OS-EXT-IPS:type") == "floating":
            floating_ip = addr["addr"]

print(f"Floating IP: {floating_ip}")
print(f"")
print(f"  MinIO Console   ->  http://{floating_ip}:9001  (admin / paperless_minio)")
print(f"  Adminer (PG)    ->  http://{floating_ip}:5050  (user / paperless_postgres)")
print(f"  Redpanda UI     ->  http://{floating_ip}:8090")
print(f"  Qdrant          ->  http://{floating_ip}:6333/dashboard")
print(f"")
print(f"  SSH: ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")

---
# Part 2: Persistent Object Storage (CHI@TACC)

The VM's local MinIO is ephemeral — it dies when the VM is deleted.
For the persistent storage deliverable, we use **CHI@TACC object storage**,
which is Chameleon's managed S3-compatible store that persists independently of VMs.

## Step 9 — Generate S3 credentials for CHI@TACC

In [ ]:
from openstack import connection
from chi import context
import chi

context.choose_project()
context.choose_site(default="CHI@TACC")

In [ ]:
conn = chi.clients.connection()

project_id = conn.current_project_id
identity_ep = conn.session.get_endpoint(service_type="identity", interface="public")
url = f"{identity_ep}/v3/users/{conn.current_user_id}/credentials/OS-EC2"

resp = conn.session.post(url, json={"tenant_id": project_id})
resp.raise_for_status()
ec2 = resp.json()["credential"]

chi_access_key = ec2["access"]
chi_secret_key = ec2["secret"]

print(f"EC2 Access: {chi_access_key}")
print(f"EC2 Secret: {chi_secret_key}")
print()
print("SAVE THESE — you will need them for the make targets.")

## Step 10 — Create the persistent bucket

Create the object storage container (bucket) at CHI@TACC.
This bucket persists even if the VM is deleted.

In [ ]:
CHI_BUCKET = f"paperless-chi-{username}"

os_conn = chi.clients.connection()
token = os_conn.authorize()
storage_url = os_conn.object_store.get_endpoint()

import swiftclient
swift_conn = swiftclient.Connection(
    preauthurl=storage_url,
    preauthtoken=token,
    retries=5
)

swift_conn.put_container(CHI_BUCKET)
print(f"Created persistent bucket: {CHI_BUCKET}")
print(f"Browse it in Horizon: https://chi.tacc.chameleoncloud.org > Object Store > Containers")

## Step 11 — Set credentials on the VM

Export the S3 credentials on the VM so `make chi-ingest` can use them.

In [ ]:
# Write credentials to a file on the VM for make targets to use
s.execute(f"""cat > ~/paperless_data/.env.chi << 'EOF'
export CHI_ACCESS_KEY={chi_access_key}
export CHI_SECRET_KEY={chi_secret_key}
export CHI_BUCKET={CHI_BUCKET}
EOF
""")
print("Credentials written to ~/paperless_data/.env.chi")
print()
print("To run the persistent ingestion pipeline, SSH in and run:")
print(f"  cd ~/paperless_data")
print(f"  source .env.chi")
print(f"  make chi-ingest")

## Step 12 — Run ingestion to persistent storage

This uploads IAM + SQuAD datasets directly to CHI@TACC persistent storage.
Course staff can browse the bucket in the Horizon GUI without needing VM access.

In [ ]:
s.execute(f"cd ~/paperless_data && source .env.chi && sg docker -c 'make chi-ingest-iam'")
print("IAM ingestion to CHI@TACC complete.")

In [ ]:
s.execute(f"cd ~/paperless_data && source .env.chi && sg docker -c 'make chi-ingest-squad'")
print("SQuAD ingestion to CHI@TACC complete.")

In [ ]:
s.execute(f"cd ~/paperless_data && source .env.chi && sg docker -c 'make chi-augment-iam'")
print("IAM augmentation to CHI@TACC complete.")

## Step 13 — Verify in Horizon

Open the CHI@TACC Horizon GUI:
1. Go to https://chi.tacc.chameleoncloud.org
2. Navigate to **Object Store > Containers**
3. Click on your bucket (e.g., `paperless-chi-xz5039-nyu-edu`)
4. You should see `warehouse/iam_dataset/` and `warehouse/squad_dataset/` with Parquet files

---
# Teardown

Run this when you are done to release VM resources.

**Note:** The CHI@TACC bucket is NOT deleted here — it persists for grading.

In [ ]:
# Uncomment and run when done
# context.choose_site(default="KVM@TACC")
# s = server.get_server(f"node-paperless-{username}")
# server.delete_server(s.id)
# l = lease.get_lease(f"lease-paperless-{username}")
# lease.delete_lease(l.id)
# print("VM resources released. CHI@TACC bucket still available for grading.")